In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import sys, os
from os.path import exists
sys.path.append('../..')

In [3]:
import numpy as np
import pylab as plt
from loguru import logger

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.monitor import Monitor

In [4]:
from vimms.Common import POSITIVE, load_obj, save_obj
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler, MZMLChromatogramSampler, RTAndIntensitySampler

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.features import obs_to_dfs
from vimms_gym.evaluation import evaluate
from vimms_gym.policy import fullscan_policy, random_policy, topN_policy, best_ppo_policy
from vimms_gym.evaluation import evaluate, run_method

# 1. Parameters

In [5]:
n_chemicals = (5000, 20000)
mz_range = (70, 1000)
rt_range = (0, 1440)
intensity_range = (1E4, 1E20)

In [6]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [7]:
isolation_window = 0.7
N = 10
rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

enable_spike_noise = True
noise_density = 0.1
noise_max_val = 1E3

In [8]:
mzml_filename = '../fullscan_QCB.mzML'    
samplers = None
samplers_pickle = 'samplers_%s.p' % mzml_filename
if exists(samplers_pickle):
    logger.info('Loaded %s' % samplers_pickle)
    samplers = load_obj(samplers_pickle)
    mz_sampler = samplers['mz']
    ri_sampler = samplers['rt_intensity']
    cr_sampler = samplers['chromatogram']
else:    
    logger.info('Creating samplers from %s' % mzml_filename)
    mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
    ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                           min_log_intensity=min_log_intensity,
                                           max_log_intensity=max_log_intensity)
    cr_sampler = MZMLChromatogramSampler(mzml_filename)
    samplers = {
        'mz': mz_sampler,
        'rt_intensity': ri_sampler,
        'chromatogram': cr_sampler
    }
    save_obj(samplers, samplers_pickle)

2022-04-04 23:41:18.657 | INFO     | __main__:<cell line: 4>:5 - Loaded samplers_fullscan_QCB.mzML.p


In [9]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
        'cr_sampler': cr_sampler,
    },
    'noise': {
        'enable_spike_noise': enable_spike_noise,
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [10]:
max_peaks = 100
in_dir = 'results_%d_%d' % (max_peaks, rt_tol)

In [11]:
total_timesteps = 20E6
n_eval_episodes = 10
deterministic = True

# 2. Training

## Train PPO

In [12]:
model_name = 'PPO'

In [13]:
# default parameters
learning_rate = 0.0003
batch_size = 64
n_steps = 2048
ent_coef = 0.0
gamma = 0.99
gae_lambda = 0.95

# modified parameters
# learning_rate = 0.0003
# batch_size = 64
# n_steps = 2048
# ent_coef = 0.01
# gamma = 0.80
# gae_lambda = 0.80

In [14]:
env = DDAEnv(max_peaks, params)
check_env(env)

### Multi-processing

In [15]:
from typing import Callable
import multiprocessing

def make_env(rank, seed=0):
    def _init():
        env = DDAEnv(max_peaks, params)
        env.seed(rank)
        env = Monitor(env)        
        return env
    set_random_seed(seed)
    return _init

num_cpu = multiprocessing.cpu_count()-1
num_cpu

191

In [16]:
if num_cpu > 100:
    num_cpu = 100

In [17]:
env = SubprocVecEnv([make_env(i) for i in range(num_cpu)]) 

In [18]:
tensorboard_log = './%s/%s_DDAEnv_tensorboard' % (in_dir, model_name)

model = PPO("MultiInputPolicy", env, 
            tensorboard_log=tensorboard_log, 
            learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
            ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda)
model.learn(total_timesteps=total_timesteps)

In [19]:
# model = PPO("MultiInputPolicy", env, verbose=2, 
#             learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
#             ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda)
# model.learn(total_timesteps=total_timesteps, log_interval=1)

In [20]:
fname = '%s/model_%s' % (in_dir, model_name)
model.save(fname)

In [21]:
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=n_eval_episodes, 
                                          deterministic=deterministic)
mean_reward, std_reward

(567.7018873, 20.634795469109125)

# 2. Evaluation

Generate some chemical sets

In [22]:
n_eval_episodes = 1
eval_dir = 'evaluation'
methods = [
    'PPO',
    'TopN',
    'random',    
]

In [23]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(n_eval_episodes):
    chems = generate_chemicals(chemical_creator_params)
    print(len(chems))
    chem_list.append(chems)

18835


Run different methods

In [24]:
max_peaks

100

In [25]:
out_dir = '%s/eval_%d_%d' % (eval_dir, max_peaks, rt_tol)
in_dir, out_dir

('results_100_15', 'evaluation/eval_100_15')

In [26]:
copy_params = dict(params)
copy_params['env']['rt_tol'] = rt_tol

for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    if method == 'PPO':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = PPO.load(fname)
    elif method == 'DQN':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = DQN.load(fname)
    else:
        model = None

    env = DDAEnv(max_peaks, copy_params)
    run_method(env, chem_list, method, out_dir, min_ms1_intensity=min_ms1_intensity, model=model, N=10, print_eval=True)
    print()

method = PPO max_peaks = 100 rt_tol = 15

Episode 0 finished after 4820 timesteps with reward 597.1519114503187
{'coverage_prop': '0.092', 'intensity_prop': '0.084', 'ms1/ms2 ratio': '0.975', 'efficiency': '0.707'}

method = TopN max_peaks = 100 rt_tol = 15

Episode 0 finished after 6084 timesteps with reward 561.4792225490279
{'coverage_prop': '0.122', 'intensity_prop': '0.110', 'ms1/ms2 ratio': '0.225', 'efficiency': '0.461'}

method = random max_peaks = 100 rt_tol = 15

Episode 0 finished after 7136 timesteps with reward 84.390974825064
{'coverage_prop': '0.033', 'intensity_prop': '0.029', 'ms1/ms2 ratio': '0.009', 'efficiency': '0.088'}

